# Brain Tumor Segmentation - U-Net++ (Inception backbone + FPN neck)

A from-scratch pixel-segmentation model on **BraTS 2021**, organised in the backbone / neck / head style:

| Structure | This model | Role |
|-----------|-----------|------|
| **Backbone** (encoder) | Inception blocks | multi-scale feature extraction |
| **Neck** (bridge) | FPN (Feature Pyramid Network) | fuse features across scales |
| **Head** (decoder) | U-Net++ nested dense skips | produce the final segmentation mask |

**Loss:** Dice loss. **Evaluation:** Dice coefficient + IoU (Jaccard), per region.
**Regions:** WT = labels 1,2,4 | TC = 1,4 | ET = 4.
Same dataset and `D:\` folder structure as before; runs on the RTX 5070.

## Setup (Miniconda + RTX 5070)
In Anaconda Prompt, once:
```bat
conda activate brats
pip install torch torchvision --index-url https://download.pytorch.org/whl/cu128
pip install nibabel opencv-python matplotlib tqdm torchinfo scipy notebook ipykernel
```
Then open this notebook in VS Code and pick the **brats** kernel. If the GPU check fails with an sm_120 error, install the nightly cu128 build instead.

### Imports & configuration
**Output:** library versions and the run settings.

In [ ]:
import os, glob, random
import numpy as np
import nibabel as nib
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
try:
    from torchinfo import summary as torchinfo_summary
    HAS_TORCHINFO = True
except Exception:
    HAS_TORCHINFO = False

# ---- paths ----
DATA_ROOTS = [r"D:\archive", "/kaggle/input"]     # first one that has patients wins
OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "./brats_out"

# ---- data ----
NUM_PATIENTS = 350
N_VAL_PATIENTS = 70            # 20%
N_TEST_PATIENTS = 70           # 20%   (train = 210 = 60%)
IMG_SIZE = 128
NEIGHBOR_SLICES = 2            # keep +-2 slices around each tumor slice
KEEP_EMPTY_FRACTION = 0.10
MAX_SLICES_PER_PATIENT = 100
MIN_BRAIN_FRACTION = 0.02
USE_FLOAT32 = True

# ---- model / training ----
BASE = 16                     # encoder base channels (lightweight)
FPN = 64                      # FPN / decoder channel width
EPOCHS = 50
PATIENCE = 15                 # early stopping
BATCH_SIZE = 16
LR = 1e-3
WEIGHT_DECAY = 1e-4
SEED = 42

MODALITIES = ["flair", "t1", "t1ce", "t2"]         # 4 input channels
REGIONS = ["WT", "TC", "ET"]
IMG_DTYPE = np.float32 if USE_FLOAT32 else np.float16

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
os.makedirs(OUT_DIR, exist_ok=True)

print("torch", torch.__version__, "| torchinfo:", HAS_TORCHINFO)
print(f"Patients {NUM_PATIENTS}: {NUM_PATIENTS-N_VAL_PATIENTS-N_TEST_PATIENTS} train / "
      f"{N_VAL_PATIENTS} val / {N_TEST_PATIENTS} test (60/20/20)")
print(f"img={IMG_SIZE} | batch={BATCH_SIZE} | epochs={EPOCHS} (patience {PATIENCE}) | LR={LR}")
print("Backbone=Inception | Neck=FPN | Head=U-Net++ | Loss=Dice | Metrics=Dice+IoU")

### GPU check
**Output:** GPU name and a real compute test.

In [ ]:
def get_device():
    if not torch.cuda.is_available():
        print("[warn] CUDA not available -> CPU (slow)."); return torch.device("cpu")
    print("GPU:", torch.cuda.get_device_name(0), "(sm_%d%d)" % torch.cuda.get_device_capability(0))
    try:
        _ = torch.randn(1024, 1024, device="cuda") @ torch.randn(1024, 1024, device="cuda")
        torch.cuda.synchronize(); print("GPU compute test: OK")
    except RuntimeError as e:
        raise SystemExit("GPU compute failed - install the cu128 build.\n" + str(e))
    return torch.device("cuda")

device = get_device()

## 1. Load Dataset
Find every BraTS patient folder (each contains `*_flair/_t1/_t1ce/_t2/_seg.nii.gz`). Unzip `BraTS2021_Training_Data.zip` into `D:\archive` first.
**Output:** number of patients found.

In [ ]:
def find_patient_dirs(roots):
    for root in roots:
        if not os.path.isdir(root):
            continue
        flairs = glob.glob(os.path.join(root, "**", "*_flair.nii.gz"), recursive=True)
        dirs = sorted({os.path.dirname(f) for f in flairs})
        if dirs:
            print(f"Found {len(dirs)} patients under {root}")
            return dirs
    raise SystemExit("No patients found. Unzip the data into D:\\archive or edit DATA_ROOTS.")

patients = find_patient_dirs(DATA_ROOTS)
print("Example:", os.path.basename(patients[0]))

## 2. Dataset structure, its channels, and their importance
Each patient is a set of co-registered 3D volumes (240x240x155). Four are MRI **modalities** (the input channels); one is the expert **segmentation**.

| Channel | Modality | What it shows | Why it matters |
|--------|----------|---------------|----------------|
| 0 | **FLAIR** | fluid-attenuated | best for the **whole tumor** / edema |
| 1 | **T1** | anatomy | structural reference |
| 2 | **T1ce** | T1 + contrast | highlights the **enhancing tumor** & core |
| 3 | **T2** | fluid-sensitive | edema and tumor extent |
| - | **seg** | labels 1/2/4 | ground-truth mask (target) |

**Output:** volume shapes, label values, and a visual of the four modalities + segmentation for one slice.

In [ ]:
def load_vol(pdir, modality):
    pid = os.path.basename(pdir)
    return nib.load(os.path.join(pdir, f"{pid}_{modality}.nii.gz")).get_fdata().astype(np.float32)

demo = patients[0]
vols = {m: load_vol(demo, m) for m in MODALITIES}
seg = load_vol(demo, "seg")
print("Patient:", os.path.basename(demo))
for m in MODALITIES:
    print(f"  {m:5s} volume shape {vols[m].shape}")
print("  seg   volume shape", seg.shape, "| label values:", np.unique(seg).astype(int).tolist())

# pick an axial slice with tumor and show all channels
z = max(range(seg.shape[2]), key=lambda k: (seg[:, :, k] > 0).sum())
fig, ax = plt.subplots(1, 5, figsize=(15, 3))
for j, m in enumerate(MODALITIES):
    ax[j].imshow(vols[m][:, :, z], cmap="gray"); ax[j].set_title(m); ax[j].axis("off")
ax[4].imshow(seg[:, :, z], cmap="nipy_spectral"); ax[4].set_title("segmentation"); ax[4].axis("off")
plt.suptitle(f"Axial slice z={z}"); plt.tight_layout(); plt.show()

## 3. Preprocessing (channel-based)
Each modality is normalised **independently** (its own brain-region mean/std), because the four channels have very different intensity ranges - normalising them together would let the brightest modality dominate. Steps:
1. **Per-channel z-score** on brain voxels (background stays 0).
2. Keep tumor slices + a few neighbours (context) + a small fraction of empty slices.
3. Resize each slice to 128x128; build the 3 target masks (WT/TC/ET).

**Output:** per-channel mean/std before vs after normalisation for one patient.

In [ ]:
def znorm_channel(vol):
    """Channel-based z-score: normalise this one modality on its brain voxels."""
    brain = vol > 0
    if brain.sum() == 0:
        return vol
    vol = (vol - vol[brain].mean()) / (vol[brain].std() + 1e-8)
    vol[~brain] = 0.0
    return vol

# show the effect of channel-based normalisation
print("Channel-based normalisation (brain voxels):")
print("Modality |  raw mean / std   -> normalised mean / std")
for m in MODALITIES:
    raw = vols[m]; b = raw > 0
    nz = znorm_channel(raw.copy()); bn = nz != 0
    print(f"  {m:5s}  | {raw[b].mean():8.1f} / {raw[b].std():7.1f} -> {nz[bn].mean():5.2f} / {nz[bn].std():4.2f}")

### Build the 2D slices (per patient) and the train/val/test arrays
**Output:** patient + slice counts per split and RAM used.

In [ ]:
def rz_img(sl): return cv2.resize(sl, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
def rz_msk(sl): return cv2.resize(sl.astype(np.uint8), (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)

def select_slices(seg, mods, Z):
    brain_ok = [(mods[0][:, :, z] != 0).mean() >= MIN_BRAIN_FRACTION for z in range(Z)]
    tumor = [z for z in range(Z) if brain_ok[z] and seg[:, :, z].max() > 0]
    ctx = set()
    for tz in tumor:
        for dz in range(-NEIGHBOR_SLICES, NEIGHBOR_SLICES + 1):
            zz = tz + dz
            if 0 <= zz < Z and brain_ok[zz]:
                ctx.add(zz)
    ctx -= set(tumor)
    empties = [z for z in range(Z) if brain_ok[z] and z not in set(tumor) | ctx]
    random.shuffle(empties); empties = empties[:int(len(empties) * KEEP_EMPTY_FRACTION)]
    keep = list(tumor) + sorted(ctx) + empties
    if MAX_SLICES_PER_PATIENT and len(keep) > MAX_SLICES_PER_PATIENT:
        keep = keep[:MAX_SLICES_PER_PATIENT]
    return sorted(set(keep))

def build_patient(pdir):
    mods = [znorm_channel(load_vol(pdir, m)) for m in MODALITIES]   # per-channel norm
    seg = load_vol(pdir, "seg"); Z = seg.shape[2]
    imgs, masks = [], []
    for z in select_slices(seg, mods, Z):
        img = np.stack([rz_img(m[:, :, z]) for m in mods], 0).astype(IMG_DTYPE)
        s = seg[:, :, z]
        mask = np.stack([rz_msk(np.isin(s, [1, 2, 4])), rz_msk(np.isin(s, [1, 4])), rz_msk(s == 4)], 0).astype(np.uint8)
        imgs.append(img); masks.append(mask)
    return (np.stack(imgs), np.stack(masks)) if imgs else (None, None)

def build_dataset(pdirs):
    pdirs = list(pdirs); random.shuffle(pdirs); pdirs = pdirs[:NUM_PATIENTS]
    val_set = set(pdirs[:N_VAL_PATIENTS]); test_set = set(pdirs[N_VAL_PATIENTS:N_VAL_PATIENTS + N_TEST_PATIENTS])
    buckets = {"train": [[], []], "val": [[], []], "test": [[], []]}; pc = {"train": 0, "val": 0, "test": 0}
    for pdir in tqdm(pdirs, desc="Preprocessing patients"):
        imgs, masks = build_patient(pdir)
        if imgs is None: continue
        split = "val" if pdir in val_set else "test" if pdir in test_set else "train"
        buckets[split][0].append(imgs); buckets[split][1].append(masks); pc[split] += 1
    out = {k: (np.concatenate(xs), np.concatenate(ys)) for k, (xs, ys) in buckets.items()}
    return out, pc

data, pc = build_dataset(patients)
(Xtr, Ytr), (Xva, Yva), (Xte, Yte) = data["train"], data["val"], data["test"]
ram = (Xtr.nbytes + Xva.nbytes + Xte.nbytes) / 1e9
for k in ["train", "val", "test"]:
    X = data[k][0]; print(f"{k.upper():5s}: {pc[k]:3d} patients -> {len(X):6d} slices")
print(f"Images in RAM: ~{ram:.1f} GB ({Xtr.dtype}). If MemoryError: lower MAX_SLICES_PER_PATIENT or set USE_FLOAT32=False.")

### Datasets + DataLoaders (augment training only)
**Output:** one training batch shape.

In [ ]:
class SliceDS(Dataset):
    def __init__(self, X, Y, augment=False):
        self.X, self.Y, self.augment = X, Y, augment
    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        x = self.X[i].astype(np.float32); y = self.Y[i].astype(np.float32)
        if self.augment:
            if random.random() < 0.5: x = x[:, :, ::-1]; y = y[:, :, ::-1]
            if random.random() < 0.5: x = x[:, ::-1, :]; y = y[:, ::-1, :]
            k = random.randint(0, 3)
            if k: x = np.rot90(x, k, (1, 2)); y = np.rot90(y, k, (1, 2))
            x = x * random.uniform(0.9, 1.1) + random.uniform(-0.1, 0.1)
            x, y = np.ascontiguousarray(x), np.ascontiguousarray(y)
        return torch.from_numpy(x.copy()), torch.from_numpy(y.copy())

pin = device.type == "cuda"
train_loader = DataLoader(SliceDS(Xtr, Ytr, True),  BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=pin)
val_loader   = DataLoader(SliceDS(Xva, Yva, False), BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=pin)
test_loader  = DataLoader(SliceDS(Xte, Yte, False), BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=pin)
xb, yb = next(iter(train_loader))
print("Batch images:", tuple(xb.shape), "| masks:", tuple(yb.shape))

## 4-5. Model: U-Net++ with Inception backbone + FPN neck
Organised exactly like your table:

- **Backbone (encoder):** five **Inception** blocks. Each block runs 1x1, 3x3, 5x5 and pooled branches in parallel and concatenates them, so every stage sees multiple receptive-field sizes.
- **Neck (bridge / FPN):** lateral 1x1 convs bring all five encoder scales to a common width, then a top-down pathway adds coarser features into finer ones (Feature Pyramid Network), fusing information across scales.
- **Head (decoder):** a **U-Net++** nested grid of dense skip connections refines the fused features back to full resolution and outputs the mask.

**Output:** module tree, layer summary, parameter count.

In [ ]:
class Inception(nn.Module):
    """Backbone block: parallel 1x1 / 3x3 / 5x5 / pooled branches, concatenated."""
    def __init__(self, cin, cout):
        super().__init__(); assert cout % 4 == 0; c = cout // 4
        def cbr(i, o, k, p): return nn.Sequential(nn.Conv2d(i, o, k, padding=p, bias=False), nn.BatchNorm2d(o), nn.ReLU(True))
        self.b1 = cbr(cin, c, 1, 0)
        self.b2 = nn.Sequential(cbr(cin, c, 1, 0), cbr(c, c, 3, 1))
        self.b3 = nn.Sequential(cbr(cin, c, 1, 0), cbr(c, c, 5, 2))
        self.b4 = nn.Sequential(nn.MaxPool2d(3, 1, 1), cbr(cin, c, 1, 0))
    def forward(self, x):
        return torch.cat([self.b1(x), self.b2(x), self.b3(x), self.b4(x)], 1)

class DoubleConv(nn.Module):
    def __init__(self, cin, cout):
        super().__init__(); self.net = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(True),
            nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(True))
    def forward(self, x): return self.net(x)

class UNetPP_Inception_FPN(nn.Module):
    def __init__(self, in_ch=4, out_ch=3, base=16, fpn=64):
        super().__init__(); self.pool = nn.MaxPool2d(2)
        ec = [base, base * 2, base * 4, base * 8, base * 8]
        # ---------- Backbone: Inception encoder ----------
        self.e0 = Inception(in_ch, ec[0]); self.e1 = Inception(ec[0], ec[1]); self.e2 = Inception(ec[1], ec[2])
        self.e3 = Inception(ec[2], ec[3]); self.e4 = Inception(ec[3], ec[4])
        # ---------- Neck: FPN (lateral 1x1 + top-down + smooth) ----------
        self.lat = nn.ModuleList([nn.Conv2d(c, fpn, 1) for c in ec])
        self.smooth = nn.ModuleList([nn.Conv2d(fpn, fpn, 3, padding=1) for _ in ec])
        # ---------- Head: U-Net++ nested dense decoder ----------
        self.dec = nn.ModuleDict({f"x{i}{j}": DoubleConv(fpn * (j + 1), fpn)
                                  for j in range(1, 5) for i in range(0, 5 - j)})
        self.out = nn.Conv2d(fpn, out_ch, 1)

    def _up(self, x, ref):
        return F.interpolate(x, size=ref.shape[-2:], mode="bilinear", align_corners=False)

    def forward(self, x):
        # backbone
        c0 = self.e0(x); c1 = self.e1(self.pool(c0)); c2 = self.e2(self.pool(c1))
        c3 = self.e3(self.pool(c2)); c4 = self.e4(self.pool(c3)); cs = [c0, c1, c2, c3, c4]
        # neck: FPN top-down fusion
        p = [None] * 5; p[4] = self.lat[4](c4)
        for i in range(3, -1, -1):
            p[i] = self.lat[i](cs[i]) + self._up(p[i + 1], cs[i])
        p = [self.smooth[i](p[i]) for i in range(5)]
        # head: U-Net++ nested dense decoder
        X = {(i, 0): p[i] for i in range(5)}
        for j in range(1, 5):
            for i in range(0, 5 - j):
                feats = [X[(i, k)] for k in range(j)] + [self._up(X[(i + 1, j - 1)], X[(i, 0)])]
                X[(i, j)] = self.dec[f"x{i}{j}"](torch.cat(feats, 1))
        return self.out(X[(0, 4)])

model = UNetPP_Inception_FPN(4, 3, BASE, FPN).to(device)
print("=============== U-Net++ (Inception backbone + FPN neck) ===============")
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")
print("\n=============== Layer summary ===============")
if HAS_TORCHINFO:
    torchinfo_summary(model, input_size=(1, 4, IMG_SIZE, IMG_SIZE),
                      col_names=("output_size", "num_params"), depth=2)
else:
    with torch.no_grad():
        print("output:", tuple(model(torch.randn(1, 4, IMG_SIZE, IMG_SIZE, device=device)).shape))

## 6. Loss - Dice Loss
Dice loss directly optimises overlap and copes with the heavy tumor/background imbalance.
**Output:** sanity check - a perfect prediction gives loss ~ 0.

In [ ]:
class DiceLoss(nn.Module):
    def forward(self, logits, y, eps=1e-6):
        probs = torch.sigmoid(logits)
        inter = (probs * y).sum((0, 2, 3))
        denom = probs.sum((0, 2, 3)) + y.sum((0, 2, 3))
        dice = (2 * inter + eps) / (denom + eps)
        return 1 - dice.mean()

_y = (torch.rand(2, 3, 32, 32) > 0.5).float()
print("Dice loss, perfect prediction:", round(DiceLoss()((_y - 0.5) * 20, _y).item(), 4), "(expect ~0)")
print("Dice loss, random  prediction:", round(DiceLoss()(torch.randn(2, 3, 32, 32), _y).item(), 4))

## 7. Evaluation metrics - Dice coefficient + IoU
Both are overlap measures (0-1, higher is better). Accumulated over the whole set for stable, dataset-level scores.
**Output:** sanity check - perfect prediction gives Dice = IoU = 1.

In [ ]:
@torch.no_grad()
def accumulate(logits, y, inter, psum, gsum, thr=0.5):
    preds = (torch.sigmoid(logits) > thr).float()
    inter += (preds * y).sum((0, 2, 3)).cpu()
    psum += preds.sum((0, 2, 3)).cpu()
    gsum += y.sum((0, 2, 3)).cpu()
    return inter, psum, gsum

def dice_iou(inter, psum, gsum, eps=1e-6):
    dice = (2 * inter + eps) / (psum + gsum + eps)
    iou = (inter + eps) / (psum + gsum - inter + eps)
    return dice, iou

_i, _p, _g = accumulate((_y - 0.5) * 20, _y, torch.zeros(3), torch.zeros(3), torch.zeros(3))
_d, _u = dice_iou(_i, _p, _g)
print("perfect prediction -> Dice", [round(v, 3) for v in _d.tolist()], "| IoU", [round(v, 3) for v in _u.tolist()])

## Training (with early stopping)
Adam, Dice loss, mixed precision, augmented training. **Output:** per-epoch train loss, val loss, val Dice/IoU, plus curves. Best model saved to `best_model.pt`.

In [ ]:
def train_model(model, tr_loader, va_loader, device):
    torch.backends.cudnn.benchmark = True
    loss_fn = DiceLoss()
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    amp = device.type == "cuda"; scaler = torch.amp.GradScaler("cuda", enabled=amp)
    hist = {k: [] for k in ["train", "val", "dice", "iou"]}
    best = -1.0; best_epoch = 0; no_improve = 0

    for ep in range(1, EPOCHS + 1):
        model.train(); run = 0.0
        for x, y in tqdm(tr_loader, leave=False, desc=f"epoch {ep}/{EPOCHS} [train]"):
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            with torch.amp.autocast("cuda", enabled=amp):
                loss = loss_fn(model(x), y)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            run += loss.item() * x.size(0)
        tr_loss = run / len(tr_loader.dataset)

        model.eval(); run = 0.0; inter = torch.zeros(3); psum = torch.zeros(3); gsum = torch.zeros(3)
        with torch.no_grad():
            for x, y in tqdm(va_loader, leave=False, desc=f"epoch {ep}/{EPOCHS} [val]"):
                x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
                with torch.amp.autocast("cuda", enabled=amp):
                    logits = model(x); loss = loss_fn(logits, y)
                run += loss.item() * x.size(0)
                inter, psum, gsum = accumulate(logits.float(), y, inter, psum, gsum)
        va_loss = run / len(va_loader.dataset)
        dice, iou = dice_iou(inter, psum, gsum)
        mdice, miou = dice.mean().item(), iou.mean().item()
        sched.step()
        for k, v in zip(hist, [tr_loss, va_loss, mdice, miou]): hist[k].append(v)

        star = ""
        if mdice > best:
            best = mdice; best_epoch = ep; no_improve = 0
            torch.save({"model": model.state_dict(), "base": BASE, "fpn": FPN, "epoch": ep}, os.path.join(OUT_DIR, "best_model.pt"))
            star = "  <- best saved"
        else:
            no_improve += 1; star = f"  (no improve {no_improve}/{PATIENCE})"
        print(f"Epoch {ep:02d}/{EPOCHS} | train {tr_loss:.4f} | val {va_loss:.4f} | "
              f"val Dice {mdice:.3f} | val IoU {miou:.3f}{star}")
        if no_improve >= PATIENCE:
            print(f"\nEarly stopping at epoch {ep} (no improvement for {PATIENCE})."); break

    ep_range = range(1, len(hist["train"]) + 1)
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    ax[0].plot(ep_range, hist["train"], label="train loss"); ax[0].plot(ep_range, hist["val"], label="val loss")
    ax[0].set_title("Dice loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
    ax[1].plot(ep_range, hist["dice"], label="val Dice"); ax[1].plot(ep_range, hist["iou"], label="val IoU")
    ax[1].set_title("Validation Dice & IoU"); ax[1].set_ylim(0, 1); ax[1].set_xlabel("epoch"); ax[1].legend()
    fig.tight_layout(); fig.savefig(os.path.join(OUT_DIR, "training_curves.png"), dpi=120); plt.show()
    print(f"\nBest mean val Dice: {best:.3f} (epoch {best_epoch})")
    return hist

history = train_model(model, train_loader, val_loader, device)

## Test evaluation - Dice + IoU (held-out patients)
Loads the best model and scores it on the 70 test patients.
**Output:** per-region Dice and IoU.

In [ ]:
ckpt = torch.load(os.path.join(OUT_DIR, "best_model.pt"), map_location=device)
best_model = UNetPP_Inception_FPN(4, 3, ckpt["base"], ckpt["fpn"]).to(device)
best_model.load_state_dict(ckpt["model"]); best_model.eval()
print(f"Loaded best model from epoch {ckpt['epoch']}")

inter = torch.zeros(3); psum = torch.zeros(3); gsum = torch.zeros(3)
with torch.no_grad():
    for x, y in tqdm(test_loader, desc="Testing"):
        x, y = x.to(device), y.to(device)
        with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
            logits = best_model(x)
        inter, psum, gsum = accumulate(logits.float(), y, inter, psum, gsum)
dice, iou = dice_iou(inter, psum, gsum)

print("\n============= TEST RESULTS =============")
print("Region |  Dice  |  IoU")
print("-------+--------+-------")
for i, r in enumerate(REGIONS):
    print(f"{r:6s} | {dice[i]:.3f}  | {iou[i]:.3f}")
print(f"{'Mean':6s} | {dice.mean():.3f}  | {iou.mean():.3f}")

## Visualise predictions
**Output:** FLAIR / ground-truth / prediction for a few test slices, with per-slice Dice.

In [ ]:
def visualize(model, X, Y, device, n_show=6):
    model.eval()
    tumor_idx = [i for i in range(len(Y)) if Y[i, 0].sum() > 50]
    picks = random.sample(tumor_idx, min(n_show, len(tumor_idx)))
    fig, axes = plt.subplots(len(picks), 3, figsize=(9, 3 * len(picks)))
    if len(picks) == 1: axes = axes[None, :]
    for r, i in enumerate(picks):
        x = torch.from_numpy(X[i].astype(np.float32)).unsqueeze(0).to(device)
        with torch.no_grad(), torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
            pred = (torch.sigmoid(model(x))[0, 0] > 0.5).cpu().numpy()
        gt = Y[i, 0]; flair = X[i, 0].astype(np.float32)
        d = (2 * (pred * gt).sum() + 1e-6) / (pred.sum() + gt.sum() + 1e-6)
        axes[r, 0].imshow(flair, cmap="gray"); axes[r, 0].set_title("FLAIR")
        axes[r, 1].imshow(flair, cmap="gray"); axes[r, 1].imshow(gt, alpha=0.4, cmap="Reds"); axes[r, 1].set_title("Ground truth (WT)")
        axes[r, 2].imshow(flair, cmap="gray"); axes[r, 2].imshow(pred, alpha=0.4, cmap="Reds"); axes[r, 2].set_title(f"Prediction (Dice {d:.2f})")
        for a in axes[r]: a.axis("off")
    fig.tight_layout(); fig.savefig(os.path.join(OUT_DIR, "predictions.png"), dpi=120); plt.show()

visualize(best_model, Xte, Yte, device)

## Notes
- **Backbone = Inception**, **Neck = FPN**, **Head = U-Net++** - one integrated segmentation model (~1.8M params, lightweight).
- Loss = Dice; evaluation = Dice coefficient + IoU, per region (WT/TC/ET).
- YOLO's detection head is intentionally not used - it produces boxes, not masks. Only its backbone/neck/head *structure* is borrowed here.
- **Lighter/faster:** lower `BASE` or `FPN`. **Stronger:** raise them, or `EPOCHS`. **Out of memory:** lower `BATCH_SIZE` or `MAX_SLICES_PER_PATIENT`.